# KKBox Churn Prediction — DuckDB Pipeline
**Audit → Clean → Feature Engineering → LightGBM → Submission**

---

### Why DuckDB for this project

| | DuckDB | PySpark (local) |
|---|---|---|
| Setup | `pip install duckdb` | JVM + PySpark config |
| 29 GB CSV scan | Columnar reads, automatic disk spill | Full file load, shuffle overhead |
| Parquet read | Near-zero-copy, column pruning | Full deserialise to JVM objects |
| SQL | Standard SQL in the same Python process | PySpark DSL or Spark SQL via strings |
| Cost | Free, forever | Free, but resource-heavy locally |
| Multi-machine | ✗ (single node only) | ✓ |

DuckDB's execution engine is **vectorized and columnar** — it processes data in SIMD-friendly batches and spills intermediate results to disk automatically when RAM is exceeded, so the 29 GB user_logs table is never fully loaded into memory.

### Architecture
```
CSV files  ──► DuckDB (one-time Parquet conversion)  ──► Parquet on disk
           ──► DuckDB SQL (audit + clean + features)  ──► Pandas DataFrames
           ──► LightGBM (5-fold CV)                   ──► submission.csv
```


## 0 · Setup

`con = duckdb.connect()` creates an **in-memory** database with no server process. DuckDB automatically uses all available CPU cores and spills to disk when a query exceeds RAM. No JVM, no session config, no schema definitions required.

Passing a file path (`duckdb.connect('kkbox.ddb')`) would persist data between sessions, but we use Parquet files for persistence instead — keeping the pipeline stateless and reproducible.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import gc, warnings
warnings.filterwarnings('ignore')

# !pip install lightgbm duckdb   # uncomment if needed
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    log_loss, roc_auc_score, classification_report,
    confusion_matrix, roc_curve, precision_recall_curve
)

# ── DuckDB connection ─────────────────────────────────────────────────────
# In-memory — state lives in the process, Parquet files are the persistence layer.
# DuckDB parallelises automatically across all CPU cores.
con = duckdb.connect()

# Memory limit: set to ~80% of available RAM before DuckDB spills to disk.
con.execute("SET memory_limit = '12GB'")

# Uncomment to redirect the temp spill directory:
# con.execute("SET temp_directory = 'C:/tmp/duckdb'")

# ── Paths ─────────────────────────────────────────────────────────────────
DATA_DIR    = Path(r"C:\Users\JJAll\Desktop\DU DATA (local, Not Drive)\Machine Learning 2\Projet")
PARQUET_DIR = DATA_DIR / "parquet"
PARQUET_DIR.mkdir(exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
COLORS = {"churn": "#e74c3c", "stay": "#2ecc71", "main": "#3498db"}
SEED = 42
np.random.seed(SEED)

print("Setup complete ✓")
print(f"DuckDB version: {duckdb.__version__}")


## 1 · One-Time Parquet Conversion

### Why convert to Parquet first?

CSV is row-oriented text — every query must read every column and parse text into typed values. Parquet is **columnar and binary compressed**:

- DuckDB reads only the columns a query actually uses (**projection pushdown**)
- Filter conditions can skip entire row groups without reading them (**predicate pushdown**)
- Numeric data compresses 4–8× vs CSV

For user_logs, a query using only `msno`, `total_secs`, `num_100` reads ~3 of 9 columns on ~6 GB of compressed data instead of 29 GB. This is the single biggest performance gain.

This cell runs **once**. After the `_SUCCESS` sentinel files exist, all future runs skip it instantly.

In [ ]:
def csv_to_parquet(csv_files: list, out_path: Path, name: str):
    """
    Convert one or more CSVs to a single Parquet directory via DuckDB.
    Multiple files (v1 + v2) are merged with UNION ALL in a single pass.
    DuckDB auto-infers types from CSV headers — no schema definition needed.
    """
    sentinel = out_path / "_SUCCESS"
    if sentinel.exists():
        print(f"  {name}: already converted ✓")
        return

    out_path.mkdir(exist_ok=True)

    # Build UNION ALL across all input files
    union_sql = "\nUNION ALL\n".join(
        f"SELECT * FROM read_csv_auto('{f}', header=True)"
        for f in csv_files
    )

    # COPY ... TO writes columnar Parquet with Snappy compression.
    # SELECT DISTINCT deduplicates rows across v1 + v2 in the same pass.
    # ROW_GROUP_SIZE controls predicate-pushdown granularity — 100k is a good default.
    con.execute(f"""
        COPY (
            SELECT DISTINCT *
            FROM ({union_sql})
        )
        TO '{out_path}'
        (FORMAT PARQUET, COMPRESSION SNAPPY, ROW_GROUP_SIZE 100000)
    """)

    sentinel.touch()
    size_gb = sum(f.stat().st_size for f in out_path.glob("*.parquet")) / 1e9
    print(f"  {name}: done → {size_gb:.2f} GB")


print("Converting CSVs to Parquet (first run only — skip if already done)...")

csv_to_parquet([DATA_DIR / "train.csv", DATA_DIR / "train_v2.csv"],
               PARQUET_DIR / "train",             "train (v1+v2)")
csv_to_parquet([DATA_DIR / "members_v3.csv"],
               PARQUET_DIR / "members",           "members")
csv_to_parquet([DATA_DIR / "transactions.csv", DATA_DIR / "transactions_v2.csv"],
               PARQUET_DIR / "transactions",      "transactions (v1+v2)")
csv_to_parquet([DATA_DIR / "user_logs.csv", DATA_DIR / "user_logs_v2.csv"],
               PARQUET_DIR / "user_logs",         "user_logs (v1+v2)")
csv_to_parquet([DATA_DIR / "sample_submission_v2.csv"],
               PARQUET_DIR / "sample_submission", "sample_submission_v2")

print("\nConversion complete ✓  All future runs read from Parquet.")


## 2 · Audit

### How DuckDB makes large-table audits fast

`COUNT(*)` on a Parquet file reads the **row group metadata** stored in the footer — not the actual row data. This makes row counts nearly instant regardless of file size.

For null detection, a single SQL query with `COUNT(*) FILTER (WHERE col IS NULL)` per column scans the file once and computes all null counts simultaneously.

For user_logs we still sample 1% — not because DuckDB can't handle the full scan, but because 300k rows gives a statistically reliable null estimate and the result is identical.

In [ ]:
# Glob patterns — DuckDB reads all .parquet files in each directory
PATHS = {
    "train":         str(PARQUET_DIR / "train"         / "*.parquet"),
    "members":       str(PARQUET_DIR / "members"       / "*.parquet"),
    "transactions":  str(PARQUET_DIR / "transactions"  / "*.parquet"),
    "user_logs":     str(PARQUET_DIR / "user_logs"     / "*.parquet"),
    "sample_sub":    str(PARQUET_DIR / "sample_submission" / "*.parquet"),
}


def audit_duckdb(path: str, name: str, sample_pct: float = 100.0):
    """
    Tiered audit using DuckDB:
      - Row count: reads Parquet footer metadata only (instant)
      - Schema:    reads Parquet footer metadata only (instant)
      - Nulls:     single-pass query, optionally on a sample

    sample_pct: percentage of rows to sample for null detection.
                100 = full scan, 1 = 1% sample (for large tables).
    """
    print(f"\n{'='*65}")
    print(f"  {name.upper()}")
    print(f"{'='*65}")

    # Schema — reads Parquet footer only, no row data scanned
    schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()
    dtype_summary = schema.groupby("column_type")["column_name"].count().to_dict()
    print(f"  Cols    : {len(schema)}")
    print(f"  Dtypes  : {dtype_summary}")
    print(f"  Columns : {schema['column_name'].tolist()}")

    # Row count — reads row group metadata from Parquet footer (very fast)
    n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{path}')").fetchone()[0]
    print(f"  Rows    : {n:,}")

    # Null detection
    cols = schema["column_name"].tolist()
    if sample_pct < 100.0:
        # USING SAMPLE: DuckDB reservoir sampling — statistically sound
        source = f"(SELECT * FROM read_parquet('{path}') USING SAMPLE {sample_pct} PERCENT)"
        note   = f" (estimated from {sample_pct:.0f}% sample)"
    else:
        source = f"read_parquet('{path}')"
        note   = ""

    # Single SELECT with one COUNT FILTER per column — one scan, all nulls
    null_sql = ", ".join(
        f'COUNT(*) FILTER (WHERE "{c}" IS NULL) AS "{c}"' for c in cols
    )
    null_result = con.execute(f"SELECT {null_sql} FROM {source}").df().iloc[0]
    null_cols   = {c: int(v) for c, v in null_result.items() if v > 0}

    if null_cols:
        print(f"  Nulls{note}:")
        for col_name, cnt in sorted(null_cols.items(), key=lambda x: -x[1]):
            est = int(cnt / (sample_pct / 100)) if sample_pct < 100 else cnt
            pct = est / n * 100
            print(f"    {col_name:35s} ~{est:>10,}  (~{pct:.1f}%)")
    else:
        print(f"  Nulls   : None ✓{note}")

    # Preview — reads first 3 rows only
    print("\n  Preview (3 rows):")
    print(con.execute(f"SELECT * FROM read_parquet('{path}') LIMIT 3").df().to_string(index=False))


audit_duckdb(PATHS["train"],        "train (v1+v2 merged)")
audit_duckdb(PATHS["members"],      "members")
audit_duckdb(PATHS["transactions"], "transactions (v1+v2 merged)")
audit_duckdb(PATHS["user_logs"],    "user_logs (v1+v2 merged)",  sample_pct=1.0)
audit_duckdb(PATHS["sample_sub"],   "sample_submission_v2")


## 3 · Data Cleaning

### SQL views vs Spark `.withColumn()` chains

In Spark, cleaning required a chain of `.withColumn()` calls — each defining a new lazy transformation in PySpark's DSL. In DuckDB we write **plain SQL**, which is more readable, easier to debug, and executes as a single optimised query plan.

DuckDB `VIEW`s are query definitions, not materialised tables — they are re-evaluated lazily when referenced, identical to Spark's lazy DataFrames. No data is stored or moved until a query actually needs it.

### 3.1 · Members

In [ ]:
# QUALIFY is a DuckDB / modern SQL clause that filters AFTER window functions.
# It's cleaner than wrapping the whole query in a subquery with WHERE rn = 1.
con.execute(f"""
    CREATE OR REPLACE VIEW members_clean AS
    SELECT
        msno,
        city,
        registered_via,

        -- Age: replace impossible values with NULL.
        -- LightGBM routes NULLs to the best-fit split child automatically.
        CASE WHEN bd < 1 OR bd > 100 THEN NULL ELSE bd END AS bd,

        -- Gender: encode male=0, female=1, unknown=NULL
        CASE WHEN gender = 'male'   THEN 0
             WHEN gender = 'female' THEN 1
             ELSE NULL END::INTEGER AS gender_enc,

        -- Registration date: YYYYMMDD integer → DATE
        -- strptime parses the string form of the integer using the format mask.
        strptime(CAST(registration_init_time AS VARCHAR), '%Y%m%d')::DATE
            AS registration_date,

        -- Membership duration: days from registration to end of observation window.
        -- More predictive than the raw date — longer members are more loyal.
        DATE_DIFF(
            'day',
            strptime(CAST(registration_init_time AS VARCHAR), '%Y%m%d')::DATE,
            DATE '2017-04-01'
        ) AS membership_duration_days

    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY msno ORDER BY registration_init_time DESC) AS rn
        FROM read_parquet('{PATHS["members"]}')
    )
    WHERE rn = 1
      -- Drop impossible durations: negative (future registration) or > 20 years
      AND DATE_DIFF(
            'day',
            strptime(CAST(registration_init_time AS VARCHAR), '%Y%m%d')::DATE,
            DATE '2017-04-01'
          ) BETWEEN 0 AND 7300
""")

print("members_clean view created ✓")
con.execute("SELECT * FROM members_clean LIMIT 3").df()


### 3.2 · Transactions

In [ ]:
con.execute(f"""
    CREATE OR REPLACE VIEW transactions_clean AS
    SELECT
        msno,
        payment_method_id,
        payment_plan_days,
        plan_list_price,
        actual_amount_paid,
        is_auto_renew,
        is_cancel,

        -- Parse YYYYMMDD integers to proper DATE columns.
        -- CAST to VARCHAR first because strptime expects a string input.
        strptime(CAST(transaction_date       AS VARCHAR), '%Y%m%d')::DATE AS transaction_date,
        strptime(CAST(membership_expire_date AS VARCHAR), '%Y%m%d')::DATE AS membership_expire_date,

        -- Discount features: computed once here, referenced in both
        -- the snapshot (last transaction) and aggregate feature queries below.
        plan_list_price - actual_amount_paid AS discount,
        CASE WHEN plan_list_price - actual_amount_paid > 0 THEN 1 ELSE 0 END AS is_discounted

    FROM read_parquet('{PATHS["transactions"]}')
""")

print("transactions_clean view created ✓")
con.execute("SELECT * FROM transactions_clean LIMIT 3").df()


### 3.3 · Train

In [ ]:
# QUALIFY = filter after window function evaluation — cleaner than a subquery.
con.execute(f"""
    CREATE OR REPLACE VIEW train_clean AS
    SELECT msno, is_churn
    FROM read_parquet('{PATHS["train"]}')
    QUALIFY ROW_NUMBER() OVER (PARTITION BY msno ORDER BY is_churn DESC) = 1
""")

n_train = con.execute("SELECT COUNT(*) FROM train_clean").fetchone()[0]
print(f"train_clean view created ✓  ({n_train:,} users)")


## 4 · Feature Engineering

### The Parquet cache pattern

For expensive aggregations we write the result to Parquet and check for it on future runs. Simpler than Spark's `cache()` + `count()` pattern — just write once, read from Parquet thereafter.

```
First run:   DuckDB aggregation → write Parquet     (minutes)
Later runs:  read_parquet(...)                      (seconds)
```

### Safe division: `NULLIF(denominator, 0)`

All ratio features use `NULLIF(x, 0)` in the denominator. When `x = 0`, `NULLIF` returns `NULL`, which propagates through arithmetic as `NULL` rather than producing a division error or `NaN`. LightGBM routes `NULL` rows to the best-fit split child — no imputation needed.

### 4.1 · Transaction Features

Two complementary views per user:

**Last transaction snapshot** — selected with `QUALIFY ROW_NUMBER() OVER (PARTITION BY msno ORDER BY transaction_date DESC) = 1`. More reliable than `MAX()` + join because it handles ties and pulls all columns in one pass.

**Aggregate history** — `GROUP BY msno` over all transactions captures long-run patterns: cancellation frequency, payment method diversity, spend variability.

In [ ]:
TXN_PARQUET = str(PARQUET_DIR / "txn_features" / "data.parquet")

if Path(TXN_PARQUET).exists():
    print("Loading cached transaction features...")

else:
    print("Building transaction features...")
    Path(TXN_PARQUET).parent.mkdir(exist_ok=True)

    last_txn_sql = """
        SELECT
            msno,
            payment_method_id           AS last_payment_method,
            payment_plan_days           AS last_plan_days,
            plan_list_price             AS last_list_price,
            actual_amount_paid          AS last_amount_paid,
            is_auto_renew               AS last_auto_renew,
            is_cancel                   AS last_is_cancel,
            discount                    AS last_discount,
            is_discounted               AS last_is_discounted,
            -- Negative days_to_expire = already expired = strong churn signal
            DATE_DIFF('day', transaction_date, membership_expire_date) AS days_to_expire,
            -- NULLIF prevents division by zero, returns NULL instead
            discount / NULLIF(plan_list_price, 0)  AS last_discount_pct
        FROM transactions_clean
        QUALIFY ROW_NUMBER() OVER (PARTITION BY msno ORDER BY transaction_date DESC) = 1
    """

    txn_agg_sql = """
        SELECT
            msno,
            COUNT(*)                                AS txn_count,
            SUM(actual_amount_paid)                 AS total_paid,
            AVG(actual_amount_paid)                 AS avg_paid,
            COALESCE(STDDEV(actual_amount_paid), 0) AS std_paid,  -- NULL for single-row users
            AVG(payment_plan_days)                  AS avg_plan_days,
            AVG(is_auto_renew)                      AS auto_renew_pct,
            AVG(is_cancel)                          AS cancel_pct,
            SUM(is_cancel)                          AS cancel_count,
            COUNT(DISTINCT payment_method_id)       AS n_payment_methods,
            COUNT(DISTINCT payment_plan_days)       AS n_unique_plans,
            MAX(plan_list_price)                    AS max_list_price,
            MIN(plan_list_price)                    AS min_list_price,
            MAX(actual_amount_paid)                 AS max_paid,
            MIN(actual_amount_paid)                 AS min_paid,
            MAX(actual_amount_paid) - MIN(actual_amount_paid) AS price_range
        FROM transactions_clean
        GROUP BY msno
    """

    # Join snapshot + history and write to Parquet in one query
    con.execute(f"""
        COPY (
            SELECT l.*, a.* EXCLUDE (msno)
            FROM ({last_txn_sql}) l
            FULL OUTER JOIN ({txn_agg_sql}) a USING (msno)
        )
        TO '{TXN_PARQUET}' (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)
    print(f"  Saved → {TXN_PARQUET}")


# Register as a DuckDB view for use in the master join
con.execute(f"CREATE OR REPLACE VIEW txn_features AS SELECT * FROM read_parquet('{TXN_PARQUET}')")

n_u = con.execute("SELECT COUNT(*) FROM txn_features").fetchone()[0]
n_c = len(con.execute("SELECT * FROM txn_features LIMIT 1").df().columns)
print(f"✓ Transaction features: {n_u:,} users × {n_c} cols")
con.execute("SELECT * FROM txn_features LIMIT 3").df()


### 4.2 · User Log Features

The most expensive query — aggregating 30+ GB down to one row per user. DuckDB handles this with **streaming execution**: it processes Parquet row groups in batches and spills intermediate hash-table state to disk if RAM is exceeded. No chunking loop required — the engine manages it automatically.

First run takes a few minutes on a laptop. Every subsequent run reads the ~30 MB result Parquet in under a second.

In [ ]:
LOG_PARQUET = str(PARQUET_DIR / "log_features" / "data.parquet")

if Path(LOG_PARQUET).exists():
    print("Loading cached log features...")

else:
    print("Building log features — DuckDB streaming over 30+ GB Parquet...")
    print("(Takes a few minutes on first run — cached forever after)")
    Path(LOG_PARQUET).parent.mkdir(exist_ok=True)

    con.execute(f"""
        COPY (
            SELECT
                msno,

                -- ── Listening volume ─────────────────────────────────────
                COUNT(date)                          AS total_logs,
                SUM(total_secs)                      AS total_secs_sum,
                MAX(total_secs)                      AS total_secs_max,
                MIN(total_secs)                      AS total_secs_min,
                AVG(total_secs)                      AS total_secs_mean,
                -- COALESCE: STDDEV returns NULL for users with only 1 active day
                COALESCE(STDDEV(total_secs), 0.0)    AS total_secs_std,

                -- ── Play-count breakdowns ─────────────────────────────────
                -- num_25 = songs played 0-25% through, num_100 = played to completion
                SUM(num_25)   AS num_25_sum,   SUM(num_50) AS num_50_sum,
                SUM(num_75)   AS num_75_sum,   SUM(num_985) AS num_985_sum,
                SUM(num_100)  AS num_100_sum,  SUM(num_unq) AS num_unq_sum,
                AVG(num_25)   AS num_25_mean,  AVG(num_100) AS num_100_mean,
                AVG(num_unq)  AS num_unq_mean,
                SUM(num_25 + num_50 + num_75 + num_985 + num_100) AS total_plays,

                -- ── Engagement ratios ─────────────────────────────────────
                -- NULLIF(total_plays, 0) = return NULL instead of div-by-zero
                -- completion_rate: fraction of songs played to the end
                SUM(num_100) / NULLIF(SUM(num_25+num_50+num_75+num_985+num_100), 0)
                                                     AS completion_rate,
                -- skip_rate: fraction abandoned in the first 25%
                SUM(num_25)  / NULLIF(SUM(num_25+num_50+num_75+num_985+num_100), 0)
                                                     AS skip_rate,
                SUM(num_50)  / NULLIF(SUM(num_25+num_50+num_75+num_985+num_100), 0)
                                                     AS half_listen_rate,
                SUM(num_985) / NULLIF(SUM(num_25+num_50+num_75+num_985+num_100), 0)
                                                     AS near_complete_rate,
                -- diversity_ratio: unique songs / total plays
                SUM(num_unq) / NULLIF(SUM(num_25+num_50+num_75+num_985+num_100), 0)
                                                     AS diversity_ratio,
                SUM(total_secs) / NULLIF(SUM(num_25+num_50+num_75+num_985+num_100), 0)
                                                     AS avg_secs_per_song,
                SUM(num_25+num_50+num_75+num_985+num_100) / NULLIF(COUNT(date), 0)
                                                     AS daily_plays,

                -- ── Active span ──────────────────────────────────────────
                -- Long span + few logs = sporadic user = churn risk
                DATE_DIFF('day', MIN(date), MAX(date)) AS active_days_span

            FROM read_parquet('{PATHS["user_logs"]}')
            GROUP BY msno
        )
        TO '{LOG_PARQUET}' (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)
    print(f"  Saved → {LOG_PARQUET}")


con.execute(f"CREATE OR REPLACE VIEW log_features AS SELECT * FROM read_parquet('{LOG_PARQUET}')")

n_u = con.execute("SELECT COUNT(*) FROM log_features").fetchone()[0]
n_c = len(con.execute("SELECT * FROM log_features LIMIT 1").df().columns)
print(f"✓ Log features: {n_u:,} users × {n_c} cols")
con.execute("SELECT * FROM log_features LIMIT 3").df()


### 4.3 · Merge → Master Datasets

DuckDB executes the full join + column selection in a single SQL query and returns a Pandas DataFrame via its **Arrow zero-copy bridge** — much faster than Spark's `toPandas()` which serialises through Java object streams.

`LEFT JOIN` semantics: every user in train/test appears in the output. Users with no transactions or logs get `NULL` in those columns — LightGBM handles this natively.

In [ ]:
MASTER_SQL = """
    SELECT
        t.msno,
        {label_col}
        m.city, m.bd, m.gender_enc, m.registered_via, m.membership_duration_days,

        -- Transaction snapshot
        x.last_payment_method, x.last_plan_days,    x.last_list_price,
        x.last_amount_paid,    x.last_auto_renew,   x.last_is_cancel,
        x.last_discount,       x.last_is_discounted, x.last_discount_pct,
        x.days_to_expire,

        -- Transaction aggregates
        x.txn_count,      x.total_paid,     x.avg_paid,      x.std_paid,
        x.avg_plan_days,  x.auto_renew_pct, x.cancel_pct,    x.cancel_count,
        x.n_payment_methods, x.n_unique_plans,
        x.max_list_price, x.min_list_price,  x.max_paid, x.min_paid, x.price_range,

        -- Log volume
        l.total_logs, l.total_secs_sum, l.total_secs_max, l.total_secs_min,
        l.total_secs_mean, l.total_secs_std,

        -- Log play counts
        l.num_25_sum, l.num_50_sum, l.num_75_sum, l.num_985_sum, l.num_100_sum,
        l.num_unq_sum, l.num_25_mean, l.num_100_mean, l.num_unq_mean, l.total_plays,

        -- Log engagement ratios
        l.completion_rate, l.skip_rate,    l.half_listen_rate,
        l.near_complete_rate, l.diversity_ratio, l.avg_secs_per_song,
        l.daily_plays, l.active_days_span

    FROM {source} t
    LEFT JOIN members_clean  m USING (msno)
    LEFT JOIN txn_features   x USING (msno)
    LEFT JOIN log_features   l USING (msno)
"""

# Train: include is_churn label
df_train = con.execute(
    MASTER_SQL.format(
        label_col="t.is_churn,",
        source=f"read_parquet('{PATHS['train']}')"
    )
).df()

# Test: no label column
df_test = con.execute(
    MASTER_SQL.format(
        label_col="",
        source=f"read_parquet('{PATHS['sample_sub']}')"
    )
).df()

print(f"df_train: {df_train.shape}")
print(f"df_test:  {df_test.shape}")

# Coverage report
print("\nMERGE COVERAGE REPORT")
print("=" * 60)
for group, col in [("members", "city"), ("transactions", "txn_count"), ("user_logs", "total_logs")]:
    tr = df_train[col].notna().mean() * 100
    te = df_test[col].notna().mean()  * 100
    print(f"  {group:15s}  train: {tr:5.1f}%  test: {te:5.1f}%")


### 4.4 · Feature Matrix

In [ ]:
DROP_COLS    = {"msno", "is_churn"}
feature_cols = [c for c in df_train.columns if c not in DROP_COLS]

X_train = df_train[feature_cols]
y_train = df_train["is_churn"]
X_test  = df_test[feature_cols]

print(f"Training:  {X_train.shape}")
print(f"Test:      {X_test.shape}")
print(f"\nFeatures ({len(feature_cols)}) — null rates:")
for c in feature_cols:
    null_tr = X_train[c].isna().mean() * 100
    null_te = X_test[c].isna().mean()  * 100
    flag = "  ⚠️" if null_tr > 30 else ""
    print(f"  {c:35s}  train {null_tr:5.1f}%   test {null_te:5.1f}%{flag}")


## 5 · Exploratory Data Analysis

EDA runs entirely in Pandas on the already-aggregated master DataFrame. The visualisation code is identical to any standard notebook — DuckDB has already done all the heavy lifting.

### 5.1 · Target Distribution

In [ ]:
n_train = len(df_train)
counts  = y_train.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(["Retained (0)", "Churned (1)"], counts.values,
              color=[COLORS["stay"], COLORS["churn"]], edgecolor="white", linewidth=1.5)
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, v + n_train * 0.005,
            f"{v:,}\n({v/n_train:.1%})", ha="center", fontweight="bold", fontsize=11)
ax.set_title("Churn Distribution", fontsize=14, fontweight="bold")
ax.set_ylabel("Count")
plt.tight_layout(); plt.show()
print(f"Churn rate: {y_train.mean():.2%}  |  Class balance: {counts[0]/counts[1]:.1f}:1")


### 5.2 · Transaction Patterns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

pd.crosstab(df_train["last_auto_renew"], y_train, normalize="index").plot(
    kind="bar", stacked=True, ax=axes[0], color=[COLORS["stay"], COLORS["churn"]], edgecolor="white")
axes[0].set_title("Churn by Auto-Renew", fontweight="bold")
axes[0].set_xticklabels(["Off", "On"], rotation=0)
axes[0].legend(["Retained", "Churned"])

pd.crosstab(df_train["last_is_cancel"], y_train, normalize="index").plot(
    kind="bar", stacked=True, ax=axes[1], color=[COLORS["stay"], COLORS["churn"]], edgecolor="white")
axes[1].set_title("Churn by Cancel Flag", fontweight="bold")
axes[1].set_xticklabels(["No Cancel", "Cancelled"], rotation=0)
axes[1].legend(["Retained", "Churned"])

for label, color, lbl in [(0, COLORS["stay"], "Retained"), (1, COLORS["churn"], "Churned")]:
    axes[2].hist(df_train[y_train == label]["last_plan_days"].dropna(),
                 bins=30, alpha=0.6, color=color, density=True, label=lbl)
axes[2].set_title("Plan Duration Distribution", fontweight="bold")
axes[2].set_xlabel("Days"); axes[2].legend()
plt.tight_layout(); plt.show()


### 5.3 · Listening Behaviour

In [ ]:
avg_daily = df_train["total_secs_sum"] / df_train["total_logs"].replace(0, 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (series, title) in zip(axes, [
    (avg_daily,                    "Avg Daily Listening (sec)"),
    (df_train["completion_rate"],  "Song Completion Rate"),
    (df_train["total_logs"],       "Number of Active Days"),
]):
    for label, color, lbl in [(0, COLORS["stay"], "Retained"),
                               (1, COLORS["churn"], "Churned")]:
        subset = series[y_train == label].dropna()
        ax.hist(subset.clip(upper=subset.quantile(0.99)),
                bins=50, alpha=0.55, color=color, density=True, label=lbl)
    ax.set_title(title, fontweight="bold"); ax.legend()
plt.tight_layout(); plt.show()


### 5.4 · Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for label, color, lbl in [(0, COLORS["stay"], "Retained"), (1, COLORS["churn"], "Churned")]:
    axes[0].hist(df_train[y_train == label]["bd"].dropna(),
                 bins=40, alpha=0.55, color=color, density=True, label=lbl)
axes[0].set_title("Age Distribution", fontweight="bold"); axes[0].legend()

pd.crosstab(df_train["gender_enc"].map({0: "Male", 1: "Female"}),
            y_train, normalize="index").plot(
    kind="bar", stacked=True, ax=axes[1], color=[COLORS["stay"], COLORS["churn"]], edgecolor="white")
axes[1].set_title("Churn by Gender", fontweight="bold")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(["Retained", "Churned"])

top_reg = df_train["registered_via"].value_counts().head(6).index
pd.crosstab(df_train[df_train["registered_via"].isin(top_reg)]["registered_via"],
            y_train, normalize="index").plot(
    kind="bar", stacked=True, ax=axes[2], color=[COLORS["stay"], COLORS["churn"]], edgecolor="white")
axes[2].set_title("Churn by Registration Channel", fontweight="bold")
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)
axes[2].legend(["Retained", "Churned"])
plt.tight_layout(); plt.show()


### 5.5 · Correlation Heatmap

In [ ]:
num_cols = [c for c in feature_cols
            if X_train[c].dtype in ["float64","int64","float32","int32"]
            and X_train[c].notna().mean() > 0.5]
tgt_corr = df_train[num_cols].corrwith(y_train).sort_values()
top_corr = pd.concat([tgt_corr.head(10), tgt_corr.tail(10)])
top12    = tgt_corr.abs().sort_values(ascending=False).head(12).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_c = [COLORS["churn"] if v > 0 else COLORS["stay"] for v in top_corr.values]
axes[0].barh(top_corr.index, top_corr.values, color=colors_c, edgecolor="white")
axes[0].axvline(0, color="black", linewidth=0.5)
axes[0].set_title("Top Feature Correlations with Churn", fontweight="bold")

corr_matrix = df_train[top12 + ["is_churn"]].corr()
sns.heatmap(corr_matrix, mask=np.triu(np.ones_like(corr_matrix, dtype=bool)),
            annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=axes[1],
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.7})
axes[1].set_title("Inter-Feature Correlations (Top 12)", fontweight="bold")
plt.tight_layout(); plt.show()


### 5.6 · Missing Values

In [ ]:
missing_pct = X_train.isnull().mean().sort_values(ascending=False) * 100
has_missing = missing_pct[missing_pct > 0]

if len(has_missing):
    fig, ax = plt.subplots(figsize=(8, max(3, len(has_missing) * 0.35)))
    colors_m = ["#e74c3c" if v > 30 else "#f39c12" if v > 10 else "#3498db"
                for v in has_missing.values]
    ax.barh(has_missing.index, has_missing.values, color=colors_m, edgecolor="white")
    ax.set_xlabel("% Missing")
    ax.set_title("Missing Values in Training Set", fontweight="bold")
    for i, (col, v) in enumerate(has_missing.items()):
        ax.text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=9)
    plt.tight_layout(); plt.show()
    print("Note: LightGBM handles NaN natively — no imputation needed.")
else:
    print("No missing values ✓")


## 6 · Model Training — LightGBM 5-Fold CV

### Parameter notes

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `learning_rate` | 0.05 | Low rate — rely on early stopping to find optimal tree count |
| `num_leaves` | 63 | ~2⁶; balances expressiveness vs overfitting |
| `max_depth` | 7 | Hard depth cap as secondary regulariser |
| `min_child_samples` | 50 | Minimum 50 samples per leaf — prevents over-splitting |
| `subsample` | 0.8 | Row subsampling per tree |
| `colsample_bytree` | 0.8 | Feature subsampling per tree |
| `n_estimators` | 1500 | Upper bound — early stopping (patience=50) finds true optimum |

`StratifiedKFold` preserves the ~7% churn rate in every fold, ensuring OOF predictions are comparable across folds and the overall OOF AUC is an unbiased generalisation estimate.

In [ ]:
params = {
    "objective":         "binary",
    "metric":            "binary_logloss",
    "boosting_type":     "gbdt",
    "learning_rate":     0.05,
    "num_leaves":        63,
    "max_depth":         7,
    "min_child_samples": 50,
    "subsample":         0.8,
    "colsample_bytree":  0.8,
    "reg_alpha":         0.1,
    "reg_lambda":        1.0,
    "n_estimators":      1500,
    "random_state":      SEED,
    "verbose":           -1,
}

N_FOLDS      = 5
skf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_preds    = np.zeros(len(X_train))
test_preds   = np.zeros(len(X_test))
feat_imp     = np.zeros(len(feature_cols))
fold_results = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    oof_preds[val_idx]  = model.predict_proba(X_val)[:, 1]
    test_preds         += model.predict_proba(X_test)[:, 1] / N_FOLDS
    feat_imp           += model.feature_importances_        / N_FOLDS

    fold_auc  = roc_auc_score(y_val, oof_preds[val_idx])
    fold_loss = log_loss(y_val, oof_preds[val_idx])
    fold_results.append({"fold": fold, "auc": fold_auc,
                         "logloss": fold_loss, "best_iter": model.best_iteration_})
    print(f"  Fold {fold}: AUC = {fold_auc:.5f}   LogLoss = {fold_loss:.5f}"
          f"   (best_iter = {model.best_iteration_})")

oof_auc  = roc_auc_score(y_train, oof_preds)
oof_loss = log_loss(y_train, oof_preds)
print(f"\n{'='*55}")
print(f"  Overall OOF AUC:     {oof_auc:.5f}")
print(f"  Overall OOF LogLoss: {oof_loss:.5f}")
print(f"{'='*55}")


## 7 · Evaluation

### 7.1 · CV Summary

In [ ]:
results_df = pd.DataFrame(fold_results)
print(results_df.to_string(index=False))
print(f"\nMean AUC:     {results_df['auc'].mean():.5f} ± {results_df['auc'].std():.5f}")
print(f"Mean LogLoss: {results_df['logloss'].mean():.5f} ± {results_df['logloss'].std():.5f}")


### 7.2 · ROC & Precision-Recall Curves

Both curves use **OOF predictions** made on each user when they were in the held-out fold. This is an unbiased generalisation estimate without needing a separate holdout set.

The Precision-Recall curve is more informative than ROC for imbalanced targets — the dashed baseline represents a random classifier at the churn rate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

fpr, tpr, _ = roc_curve(y_train, oof_preds)
axes[0].plot(fpr, tpr, color=COLORS["main"], lw=2, label=f"AUC = {oof_auc:.4f}")
axes[0].plot([0,1],[0,1], "k--", alpha=0.3)
axes[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC Curve (OOF)")
axes[0].legend(fontsize=12)

prec, rec, _ = precision_recall_curve(y_train, oof_preds)
axes[1].plot(rec, prec, color=COLORS["churn"], lw=2)
axes[1].axhline(y_train.mean(), color="gray", linestyle="--", alpha=0.5,
                label=f"Baseline = {y_train.mean():.3f}")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall Curve (OOF)")
axes[1].legend(fontsize=11)
plt.tight_layout(); plt.show()


### 7.3 · Confusion Matrix

In [ ]:
# Threshold 0.5 by default. Lowering it (e.g. to 0.3) catches more churners
# at the cost of more false positives — adjust based on business cost of each error type.
oof_labels = (oof_preds >= 0.5).astype(int)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_train, oof_labels), annot=True, fmt=",d",
            cmap="Blues", ax=ax,
            xticklabels=["Retained", "Churned"],
            yticklabels=["Retained", "Churned"])
ax.set(xlabel="Predicted", ylabel="Actual",
       title="Confusion Matrix (OOF, threshold = 0.5)")
plt.tight_layout(); plt.show()
print(classification_report(y_train, oof_labels, target_names=["Retained", "Churned"]))


### 7.4 · Feature Importance

In [ ]:
# Averaged across 5 folds to reduce variance from any single split.
imp_df = (pd.DataFrame({"feature": feature_cols, "importance": feat_imp})
          .sort_values("importance", ascending=True))

fig, ax = plt.subplots(figsize=(8, 8))
top25 = imp_df.tail(25)
ax.barh(top25["feature"], top25["importance"], color=COLORS["main"], edgecolor="white")
ax.set(xlabel="Importance (avg split gain)", title="Top 25 Features — Avg across 5 Folds")
plt.tight_layout(); plt.show()


### 7.5 · Prediction Distributions

In [ ]:
# A well-calibrated model shows clear separation between the two OOF distributions.
# If the test distribution looks very different from OOF, suspect a train/test
# distribution shift — common in time-series competitions like KKBox.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for label, color, lbl in [(0, COLORS["stay"], "Retained"), (1, COLORS["churn"], "Churned")]:
    axes[0].hist(oof_preds[y_train == label], bins=80,
                 alpha=0.6, color=color, density=True, label=lbl)
axes[0].axvline(0.5, color="black", linestyle="--", alpha=0.5, label="Threshold = 0.5")
axes[0].set(title="OOF Prediction Distribution", xlabel="Predicted churn probability")
axes[0].legend()

axes[1].hist(test_preds, bins=80, color=COLORS["main"], alpha=0.7, edgecolor="white")
axes[1].axvline(test_preds.mean(), color="red", linestyle="--",
                label=f"Mean = {test_preds.mean():.3f}")
axes[1].set(title="Test Prediction Distribution", xlabel="Predicted churn probability")
axes[1].legend()
plt.tight_layout(); plt.show()


## 8 · Generate Submission

In [ ]:
submission = con.execute(
    f"SELECT msno FROM read_parquet('{PATHS[\"sample_sub\"]}')"
).df()
submission["is_churn"] = test_preds

OUT_PATH = DATA_DIR / "submission_duckdb_v1.csv"
submission.to_csv(OUT_PATH, index=False)

print(f"Saved → {OUT_PATH}")
print(f"Shape:  {submission.shape}")
print("\nPrediction stats:")
print(submission["is_churn"].describe())

con.close()
print("\n✅ DuckDB pipeline complete.")


---
## Summary

### What DuckDB replaced and why

| Spark pattern | DuckDB equivalent | Improvement |
|---|---|---|
| Explicit schema `StructType` | Auto-inferred from Parquet metadata | No schema definition needed |
| `repartition(400, 'msno')` | Automatic parallelism across all cores | No tuning required |
| `.cache()` + `.count()` to materialise | Parquet write + read | Simpler, no JVM memory pressure |
| `Window.partitionBy().orderBy()` | `QUALIFY ROW_NUMBER() OVER (...)` | Same logic, standard SQL |
| `F.coalesce(F.col('x'), F.lit(0.0))` | `COALESCE(x, 0.0)` | Standard SQL |
| Chunked user_logs audit loop | `USING SAMPLE 1 PERCENT` | One SQL clause |
| `.withColumn()` cleaning chains | `CREATE VIEW AS SELECT ... CASE WHEN` | Readable SQL |
| `.toPandas()` via JVM serialisation | `.df()` via Arrow zero-copy | Faster, less memory |
| `F.when(...).otherwise(None)` | `NULLIF(x, 0)` | Concise standard SQL |

### Key predictors (typical ranking)
1. `last_is_cancel` — explicit cancellation, strongest single signal
2. `last_auto_renew` — passive churn risk when off
3. `days_to_expire` — proximity to expiry at last transaction
4. `completion_rate` — engagement quality
5. `membership_duration_days` — loyalty signal

### Potential next steps
- **Optuna** hyperparameter search over `num_leaves`, `learning_rate`, `min_child_samples`
- **SHAP values** for per-user interpretability (`shap.TreeExplainer`)
- **Time-decay features** — weight recent logs more heavily (e.g. last-30-day `completion_rate` vs all-time)
- **Target encoding** for `city` and `registered_via` inside each CV fold
- **Stacking** with XGBoost / CatBoost for ensemble uplift
- **MotherDuck** if you want to share this pipeline with teammates without a full cloud warehouse
